# 02 — Build retrieval eval set

Phase C: bootstrap `datasets/retrieval_eval.jsonl` — the ≥200-pair eval set the harness scores against. Plan §B step 1–5:

1. For each source, take items and build a 2–3-sentence summary (LLM, one-shot).
2. Generate candidate queries from each summary (avoiding raw-body phrasing).
3. Reject queries with >3-gram overlap with the source body — aim <5% retained verbatim phrasing.
4. Hand-curate ~30% per source — synonyms, abbreviations, real user-style questions.
5. Add 5–10 cross-source negative pairs per source.
6. Pin the file in git.

Generation is LLM-driven; ragas's `TestsetGenerator` is one option but the dependency lookup is your call. Either way, the output schema is a JSONL of `{query, source, expected_content_id}`.

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv

from domains.notes.sources import LocalFileSource
from domains.raw_store.sources import RawStoreSource
from domains.research.sources import ResearchSource
from domains.sessions.sources import SessionsSource

# Resolve repo root from cwd — Jupyter may launch from notebooks/ or repo root.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

# Load .env so OPENAI_API_KEY (etc.) are available without shell setup.
load_dotenv(REPO_ROOT / ".env")

# Required env. No silent defaults — silent defaults bit us once with an
# expanduser("~") fallback that opened an unrelated empty raw_store.db.
backup = os.environ.get("BACKUP_SOURCE_DIR")
if not backup:
    raise RuntimeError(
        "BACKUP_SOURCE_DIR is not set. Add it to .env or export in your shell."
    )
BACKUP = Path(backup).expanduser()

OUT = REPO_ROOT / "packages" / "evals" / "datasets" / "retrieval_eval.jsonl"
print(f"REPO_ROOT={REPO_ROOT}\nBACKUP={BACKUP}\nOUT={OUT}")

In [2]:
# Load items per source. Validate paths first so failures are clear.
_paths = {
    "raw_store": BACKUP / "raw_store.db",
    "sessions": BACKUP / "sessions.db",
    "research": BACKUP / "research.db",
    "notes": BACKUP / "notes",
}
missing = {k: p for k, p in _paths.items() if not p.exists()}
if missing:
    raise FileNotFoundError(f"missing under BACKUP_SOURCE_DIR: {missing}")

items_by_source = {
    "raw_store": RawStoreSource(_paths["raw_store"]).get_items(),
    "sessions": SessionsSource(_paths["sessions"]).get_items(),
    "research": ResearchSource(_paths["research"]).get_items(),
    "notes": LocalFileSource(_paths["notes"]).get_items(),
}
{k: len(v) for k, v in items_by_source.items()}

{'raw_store': 950, 'sessions': 185, 'research': 7, 'notes': 49}

## Inspect: source size distribution + session turn buckets

Sanity-check item shapes and bucket sessions by turn count for stratified sampling.

In [3]:
import re

def session_turn_count(text: str) -> int:
    return len(re.findall(r"<<<TURN", text))

# Per-source size stats
for name, items in items_by_source.items():
    lens = sorted(len(i.text or "") for i in items)
    n = len(lens)
    p50 = lens[n // 2]
    p90 = lens[int(n * 0.9)]
    print(f"{name:10s}  n={n:4d}  chars min={lens[0]:5d}  p50={p50:6d}  p90={p90:7d}  max={lens[-1]:7d}")

# Session turn-count bucketing (for stratified sampling)
turn_counts = [session_turn_count(s.text) for s in items_by_source["sessions"]]
buckets = {"short_le4": [], "mid_5_15": [], "long_gt15": []}
for s, t in zip(items_by_source["sessions"], turn_counts, strict=True):
    if t <= 4: buckets["short_le4"].append(s)
    elif t <= 15: buckets["mid_5_15"].append(s)
    else: buckets["long_gt15"].append(s)
print("\nsession turn buckets:", {k: len(v) for k, v in buckets.items()})

raw_store   n= 950  chars min=    0  p50=     0  p90=  37813  max= 233492
sessions    n= 185  chars min=    0  p50=  6354  p90=  17996  max=  57263
research    n=   7  chars min= 3610  p50=  5584  p90=  20015  max=  20015
notes       n=  49  chars min=  177  p50=  5086  p90=  18592  max=  31531

session turn buckets: {'short_le4': 22, 'mid_5_15': 60, 'long_gt15': 103}


## Sample (Option 1 targets — adapted to corpus)

Filter empties, then stratified-sample per source. Seed=42 for reproducibility.

| Source | Sampled items × queries/item = positives |
|---|---|
| raw_store | 60 × 1 = 60 (recent/older × short/long) |
| sessions  | 25 × 2 = 50 (~8 per turn-bucket) |
| notes     | 35 × 1 = 35 (stratified by char-length quartile) |
| research  | 7 × 3 = 21  (all, facet-per-query) |

In [4]:
import random
from datetime import date, timedelta

random.seed(42)

def stratified_pick(items, key_fn, target):
    """Bucket items by key_fn, then pick approximately equal counts per bucket up to target."""
    buckets: dict = {}
    for it in items:
        buckets.setdefault(key_fn(it), []).append(it)
    per_bucket = max(1, target // len(buckets))
    picked = []
    leftover_pool = []
    for k, bucket in buckets.items():
        random.shuffle(bucket)
        picked.extend(bucket[:per_bucket])
        leftover_pool.extend(bucket[per_bucket:])
    random.shuffle(leftover_pool)
    picked.extend(leftover_pool[: max(0, target - len(picked))])
    return picked[:target]

# raw_store: 60, stratified by (recency × length quartile)
rs_nonempty = [i for i in items_by_source["raw_store"] if (i.text or "").strip()]
RECENCY_CUTOFF = date(2026, 4, 10)  # ~30 days before today
def rs_key(i):
    age = "recent" if (i.date and i.date >= RECENCY_CUTOFF) else "older"
    size = "short" if len(i.text) < 8000 else "long"
    return (age, size)
sample_raw_store = stratified_pick(rs_nonempty, rs_key, target=60)

# sessions: 25, stratified by turn-count bucket
def sess_key(i):
    t = session_turn_count(i.text)
    return "short" if t <= 4 else ("mid" if t <= 15 else "long")
sess_nonempty = [i for i in items_by_source["sessions"] if (i.text or "").strip()]
sample_sessions = stratified_pick(sess_nonempty, sess_key, target=25)

# notes: 35, stratified by char-length quartile
notes_items = items_by_source["notes"]
sorted_lens = sorted(len(i.text) for i in notes_items)
q1, q2, q3 = sorted_lens[len(sorted_lens) // 4], sorted_lens[len(sorted_lens) // 2], sorted_lens[3 * len(sorted_lens) // 4]
def notes_key(i):
    n = len(i.text)
    if n < q1: return "xs"
    if n < q2: return "s"
    if n < q3: return "m"
    return "l"
sample_notes = stratified_pick(notes_items, notes_key, target=35)

# research: take all 7
sample_research = list(items_by_source["research"])

sampled = {
    "raw_store": sample_raw_store,
    "sessions": sample_sessions,
    "notes": sample_notes,
    "research": sample_research,
}
{k: len(v) for k, v in sampled.items()}

{'raw_store': 60, 'sessions': 25, 'notes': 35, 'research': 7}

## Step 1–2: LLM summarise + generate queries (gpt-4o-mini, disk-cached)

Per item: 2-sentence summary + N candidate queries (N varies by source target). Cache responses to `data/eval_cache/query_gen/{source}_{item_id}.json` so reruns are free.

Cost ballpark: ~127 sampled items × ~6k tokens in + ~250 tokens out × `gpt-4o-mini` ≈ **~$0.12 total**.

In [5]:
import hashlib
from openai import OpenAI

CACHE_DIR = REPO_ROOT / "data" / "eval_cache" / "query_gen"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# How many queries to generate per item, by source.
QUERIES_PER_ITEM = {"raw_store": 1, "sessions": 2, "notes": 1, "research": 3}
# Generate slightly more than target so the n-gram filter has slack.
GEN_OVERAGE = {"raw_store": 2, "sessions": 3, "notes": 2, "research": 5}

PROMPT = """You generate evaluation queries for a knowledge retrieval system.

Document title: {title}
Document body (truncated to 4000 chars):
---
{body}
---

1. Write a 2-sentence summary capturing what makes THIS document distinct (topic, claims, or angle).
2. Generate {n} retrieval queries a real user would type to find THIS document.
   Constraints:
   - 6-15 words each
   - Natural question or imperative phrasing
   - Do NOT quote phrases verbatim from the body
   - Vary phrasing (different angles/framings) across the {n} queries

Return ONLY a JSON object: {{"summary": "...", "queries": ["...", ...]}}"""

client = OpenAI()

def cache_path(source: str, item_id: str) -> Path:
    safe = hashlib.sha1(item_id.encode()).hexdigest()[:16]
    return CACHE_DIR / f"{source}_{safe}.json"

def gen_for_item(source: str, item, n: int) -> dict:
    cp = cache_path(source, item.item_id)
    if cp.exists():
        return json.loads(cp.read_text())
    prompt = PROMPT.format(title=(item.title or "(untitled)")[:200], body=item.text[:4000], n=n)
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0.7,
    )
    data = json.loads(resp.choices[0].message.content)
    data["_item_id"] = item.item_id
    data["_source"] = source
    data["_usage"] = {"in": resp.usage.prompt_tokens, "out": resp.usage.completion_tokens}
    cp.write_text(json.dumps(data, indent=2))
    return data

print(f"setup complete — cache at {CACHE_DIR.relative_to(REPO_ROOT)}")

setup complete — cache at data/eval_cache/query_gen


In [6]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import perf_counter

# Build work list — skip items already cached.
todo = []
for source, items in sampled.items():
    n_per = QUERIES_PER_ITEM[source] + GEN_OVERAGE[source]
    for it in items:
        if not cache_path(source, it.item_id).exists():
            todo.append((source, it, n_per))

cached_already = sum(len(items) for items in sampled.values()) - len(todo)
print(f"to generate: {len(todo)}    already cached: {cached_already}")

def gen_worker(args):
    src, it, n = args
    try:
        gen_for_item(src, it, n=n)
        return ("ok",)
    except Exception as e:
        return ("err", src, it.item_id, repr(e))

errors = []
t0 = perf_counter()
if todo:
    with ThreadPoolExecutor(max_workers=8) as ex:
        for fut in as_completed(ex.submit(gen_worker, a) for a in todo):
            r = fut.result()
            if r[0] == "err":
                errors.append(r)
print(f"elapsed: {perf_counter() - t0:.1f}s    errors: {len(errors)}")
for e in errors[:5]:
    print(" ", e)

# Load cached results back into a uniform dict + tally cost.
loaded: dict[str, list[dict]] = {s: [] for s in sampled}
total_in = total_out = 0
for source, items in sampled.items():
    for it in items:
        data = json.loads(cache_path(source, it.item_id).read_text())
        loaded[source].append(data)
        u = data.get("_usage", {})
        total_in += u.get("in", 0)
        total_out += u.get("out", 0)
cost = total_in / 1_000_000 * 0.15 + total_out / 1_000_000 * 0.60
print(f"\ncached results: {{ {', '.join(f'{s}: {len(v)}' for s, v in loaded.items())} }}")
print(f"tokens in/out: {total_in}/{total_out}    approx cost: ${cost:.4f}")

to generate: 21    already cached: 106


elapsed: 9.4s    errors: 0

cached results: { raw_store: 60, sessions: 25, notes: 35, research: 7 }
tokens in/out: 123734/12953    approx cost: $0.0263


In [7]:
# Step 3-4: n-gram filter + per-source dedup cap.

def ngram_overlap_ratio(query: str, body: str, n: int = 3) -> float:
    def grams(s):
        toks = s.lower().split()
        return {tuple(toks[i : i + n]) for i in range(len(toks) - n + 1)}
    q, b = grams(query), grams(body)
    return len(q & b) / max(len(q), 1)

OVERLAP_THRESHOLD = 0.3
MAX_PER_ITEM = {"raw_store": 1, "sessions": 2, "notes": 1, "research": 3}

# Build candidate pool: (query, source, item_id, overlap)
candidates = []
items_by_id = {
    src: {it.item_id: it for it in xs} for src, xs in items_by_source.items()
}
for source, gen_results in loaded.items():
    for r in gen_results:
        item = items_by_id[source][r["_item_id"]]
        for q in r["queries"]:
            ov = ngram_overlap_ratio(q, item.text)
            candidates.append({"query": q, "source": source, "expected_content_id": item.item_id, "overlap": ov})

# Filter on overlap, then per-item cap (keep lowest-overlap queries first).
survivors = []
per_item_count: dict[tuple[str, str], int] = {}
candidates.sort(key=lambda c: c["overlap"])  # best first
for c in candidates:
    if c["overlap"] > OVERLAP_THRESHOLD:
        continue
    key = (c["source"], c["expected_content_id"])
    if per_item_count.get(key, 0) >= MAX_PER_ITEM[c["source"]]:
        continue
    per_item_count[key] = per_item_count.get(key, 0) + 1
    survivors.append(c)

# Per-source counts after filter+dedup
by_source = {}
for s in survivors:
    by_source[s["source"]] = by_source.get(s["source"], 0) + 1
print("after filter+dedup:", by_source)
print(f"total candidates: {len(candidates)}, kept: {len(survivors)}, "
      f"reject_overlap: {sum(1 for c in candidates if c['overlap'] > OVERLAP_THRESHOLD)}")

after filter+dedup: {'raw_store': 60, 'sessions': 50, 'notes': 35, 'research': 21}
total candidates: 466, kept: 166, reject_overlap: 21


## Step 5–6: write JSONL + validate

v0: ship filtered LLM output (no hand-curation, no negatives). Quality polish is a follow-up cadence-review pass.

In [8]:
from evals.retrieval.dataset import group_by_source, load_eval_set

# Stable order: by source then expected_content_id then query
survivors_sorted = sorted(survivors, key=lambda s: (s["source"], s["expected_content_id"], s["query"]))

OUT.parent.mkdir(parents=True, exist_ok=True)
with OUT.open("w") as f:
    for s in survivors_sorted:
        f.write(json.dumps({
            "query": s["query"],
            "source": s["source"],
            "expected_content_id": s["expected_content_id"],
        }) + "\n")
print(f"wrote {len(survivors_sorted)} pairs to {OUT}")

# Validate via the harness's loader
pairs = load_eval_set(OUT)
print("loader OK, per-source:", {k: len(v) for k, v in group_by_source(pairs).items()})

wrote 166 pairs to /Users/cyyang/GitHub/knowledge-pipeline/packages/evals/datasets/retrieval_eval.jsonl
loader OK, per-source: {'raw_store': 60, 'notes': 35, 'sessions': 50, 'research': 21}
